# Study 820 — Expected-Shortfall Premium 🎯📉

**Do stocks with a fat left tail go on to earn *more*?**

A downside **tail-risk premium**: a stock whose recent daily returns carry a large
**Expected Shortfall** (CVaR at 5% — the mean of its *worst 5%* of days) is exposed to
deeper crashes, so if that tail risk is *priced* it should be compensated with a higher
future return. Sort long **high-ES** / short **low-ES**. We take the self-contained
daily version on a liquid US cross-section (2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — the bias flatters this
sort, so magnitudes are an upper bound.*


## 1. The idea in one picture

**Expected Shortfall** answers *'on my worst days, how bad is bad?'* — it is the average of the deepest 5% of daily losses. A fat left tail is real crash exposure; a rational market should pay you to hold it. So rank the cross-section on trailing one-year ES; buy the fat-tail names, sell the calm ones, and see if the risk is compensated.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=5.41, t_nw=2.8, hi_bps=10.43, lo_bps=5.03, gross_sharpe=0.69)
print('long high-ES / short low-ES spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  high-ES book %+.2f bps vs low-ES book %+.2f bps'
      % (R['hi_bps'], R['lo_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long high-ES / short low-ES spread: +5.41 bps/day (NW t = +2.80)
  high-ES book +10.43 bps vs low-ES book +5.03 bps
  gross spread Sharpe (before cost): 0.69


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`, fat left tail → higher mean) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, tail fatness present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from expected_shortfall import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=820, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0024, seed=820, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -0.26  (should be ~0)
planted world: spread NW t = +3.54  (should light up)


## 3. The honest verdict — real in sign, but a *volatility* sort in disguise

On this liquid mega-cap tape the long-high-ES / short-low-ES spread is **+5.41 bps/day** with NW *t* = **+2.80** — the claimed sign *holds*, and the observed value sits ~4.8σ into the **right** tail of a 1,000-permutation placebo. But read the fine print: Expected Shortfall is ~collinear with volatility, so this is essentially a **long-high-vol / short-low-vol** sort — the *opposite* of the low-vol anomaly — and on a **survivor** universe it just says the high-vol tech mega-caps (NVDA, TSLA, AMD) won 2010–2026. It is era-dependent (*t* = +1.67 pre-2018 vs +2.26 after, significant in only one half). **Signal: Weak**, **Tradability: Fragile** — at 1 bp it nets only +3.27 bps/day (*t* = +1.64) and dies at 5 bps.